In [6]:
import httpx
import tomli
import pandas as pd

with open('config.toml','rb') as f:
	config = tomli.load(f)

'{"help": "https://localhost:8443/api/3/action/help_show?name=datastore_search", "success": true, "result": {"include_total": true, "limit": 100, "records_format": "objects", "resource_id": "efc9d958-e438-4d29-8db1-cdee343a9092", "total_estimation_threshold": null, "records": [{"_id":1,"dms_site_id":"CVWD 17","agency":"CVWD","local_site_id":1000541},{"_id":2,"dms_site_id":"CVWD 11","agency":"CVWD","local_site_id":1002294},{"_id":3,"dms_site_id":"CVWD 12","agency":"CVWD","local_site_id":1002289},{"_id":4,"dms_site_id":"CVWD 15","agency":"CVWD","local_site_id":1000543},{"_id":5,"dms_site_id":"CVWD 16","agency":"CVWD","local_site_id":1000547},{"_id":6,"dms_site_id":"CVWD 19","agency":"CVWD","local_site_id":1000568},{"_id":7,"dms_site_id":"CVWD 24","agency":"CVWD","local_site_id":1000569},{"_id":8,"dms_site_id":"CVWD 20","agency":"CVWD","local_site_id":1000573},{"_id":9,"dms_site_id":"CVWD 21","agency":"CVWD","local_site_id":1000570},{"_id":10,"dms_site_id":"CVWD 26","agency":"CVWD","local

In [52]:
table_id = "efc9d958-e438-4d29-8db1-cdee343a9092"
base_url = 'https://localhost:8443/api/3/action'
url = base_url + '/datastore_search'

C = httpx.Client(
	headers={'Authorization': config['CKAN']['token']},
	verify=False)

r = C.get(
	url,
	params={'resource_id':table_id},
	)
r.text

'{"help": "https://localhost:8443/api/3/action/help_show?name=datastore_search", "success": true, "result": {"include_total": true, "limit": 100, "records_format": "objects", "resource_id": "efc9d958-e438-4d29-8db1-cdee343a9092", "total_estimation_threshold": null, "records": [{"_id":1,"dms_site_id":"CVWD 17","agency":"CVWD","local_site_id":1000541},{"_id":2,"dms_site_id":"CVWD 11","agency":"CVWD","local_site_id":1002294},{"_id":3,"dms_site_id":"CVWD 12","agency":"CVWD","local_site_id":1002289},{"_id":4,"dms_site_id":"CVWD 15","agency":"CVWD","local_site_id":1000543},{"_id":5,"dms_site_id":"CVWD 16","agency":"CVWD","local_site_id":1000547},{"_id":6,"dms_site_id":"CVWD 19","agency":"CVWD","local_site_id":1000568},{"_id":7,"dms_site_id":"CVWD 24","agency":"CVWD","local_site_id":1000569},{"_id":8,"dms_site_id":"CVWD 20","agency":"CVWD","local_site_id":1000573},{"_id":9,"dms_site_id":"CVWD 21","agency":"CVWD","local_site_id":1000570},{"_id":10,"dms_site_id":"CVWD 26","agency":"CVWD","local

In [53]:
df = pd.DataFrame(r.json()['result']['records'])
print(df.shape)
df.head()

(58, 4)


,_id,dms_site_id,agency,local_site_id
0,1,CVWD 17,CVWD,1000541.0
1,2,CVWD 11,CVWD,1002294.0
2,3,CVWD 12,CVWD,1002289.0
3,4,CVWD 15,CVWD,1000543.0
4,5,CVWD 16,CVWD,1000547.0


In [50]:
url = base_url + '/group_list'
r = C.get(
	url,
	params={'resource_id':table_id},
	)
df = pd.DataFrame(r.json()['result']).rename(columns={0:'group'})
df.head(2)
# r.json()

,group
0,cucamonga_basin


In [61]:
url = base_url + '/datastore_upsert'

wells = [{
	# 'dms_site_id':'test',
	# 'agency':'text',
	# 'local_site_id':2000547.0
	}]

r = C.post(
	url,
	data={
		'resource_id':table_id,
		'records':wells,
		'method':'insert',
		'force':True,
	}
	)
print(r.url)
print(r.reason_phrase)

r.json()
# r.url

https://localhost:8443/api/3/action/datastore_upsert
CONFLICT


{'help': 'https://localhost:8443/api/3/action/help_show?name=datastore_upsert',
 'error': {'records': ['row "0" is not a json object'],
  '__type': 'Validation Error'},
 'success': False}

In [60]:
txt = httpx.get("https://localhost:8443/api/3/action/help_show?name=datastore_upsert",verify=False).json()['result']
print(txt)

Updates or inserts into a table in the DataStore

    The datastore_upsert API action allows you to add or edit records to
    an existing DataStore resource. In order for the *upsert* and *update*
    methods to work, a unique key has to be defined via the datastore_create
    action. The available methods are:

    *upsert*
        Update if record with same key already exists, otherwise insert.
        Requires unique key or _id field.
    *insert*
        Insert only. This method is faster that upsert, but will fail if any
        inserted record matches an existing one. Does *not* require a unique
        key.
    *update*
        Update only. An exception will occur if the key that should be updated
        does not exist. Requires unique key or _id field.


    :param resource_id: resource id that the data is going to be stored under.
    :type resource_id: string
    :param force: set to True to edit a read-only resource
    :type force: bool (optional, default: False)
    :par